# Copy TSV Data To S3

<img src="img/write_tsv_to_s3.png" width="45%" align="left">

#### We have chosen the [Amazon Customer Reviews Dataset](https://s3.amazonaws.com/dsoaws/amazon-reviews-pds/readme.html) as our main dataset.

The dataset is shared in a public Amazon S3 bucket: s3://aai-540/amazon-reviews-pds/tsv/

We can use the AWS Command Line Interface (CLI) to list the S3 bucket content using the following CLI commands:

In [1]:
!aws s3 ls s3://aai-540/amazon-reviews-pds/tsv/

2025-09-03 17:32:23          0 
2025-09-03 18:30:01 3238702530 amazon_reviews_us_Books_v1_02.tsv
2025-09-03 18:30:01   53855391 amazon_reviews_us_Digital_Software_v1_00.tsv
2025-09-03 18:30:01   73154460 amazon_reviews_us_Digital_Video_Games_v1_00.tsv
2025-09-03 18:30:01   39977611 amazon_reviews_us_Gift_Card_v1_00.tsv


# To Simulate an Application Writing Into Our Data Lake, We Copy the Public TSV Dataset to a Private S3 Bucket in our Account

<img src="img/copy_data_to_s3.png" width="60%" align="left">

# Check Pre-Requisites from an earlier notebook

In [2]:
%store -r setup_dependencies_passed

In [3]:
try:
    setup_dependencies_passed
except NameError:
    print("+++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL NOTEBOOKS IN THE SETUP FOLDER FIRST. You are missing Setup Dependencies.")
    print("+++++++++++++++++++++++++++++++")

In [4]:
print(setup_dependencies_passed)

True


In [5]:
%store -r setup_s3_bucket_passed

In [6]:
try:
    setup_s3_bucket_passed
except NameError:
    print("+++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL NOTEBOOKS IN THE SETUP FOLDER FIRST. You are missing Setup S3 Bucket.")
    print("+++++++++++++++++++++++++++++++")

In [7]:
print(setup_s3_bucket_passed)

True


In [8]:
if not setup_dependencies_passed:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL NOTEBOOKS IN THE SETUP FOLDER FIRST. You are missing Setup Dependencies.")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
if not setup_s3_bucket_passed:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL NOTEBOOKS IN THE SETUP FOLDER FIRST. You are missing Setup S3 Bucket.")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")

In [10]:
import boto3
import sagemaker
import pandas as pd

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")

sm = boto3.Session().client(service_name="sagemaker", region_name=region)
print(bucket)

sagemaker-us-east-1-944202758041


# Set S3 Source Location (Public S3 Bucket)

In [10]:
s3_public_path_tsv = "s3://aai-540/amazon-reviews-pds/tsv"

In [11]:
%store s3_public_path_tsv

Stored 's3_public_path_tsv' (str)


# Set S3 Destination Location (Our Private S3 Bucket)

In [12]:
s3_private_path_tsv = "s3://{}/amazon-reviews-pds/tsv".format(bucket)
print(s3_private_path_tsv)

s3://sagemaker-us-east-1-944202758041/amazon-reviews-pds/tsv


In [13]:
%store s3_private_path_tsv

Stored 's3_private_path_tsv' (str)


# Copy Data From the Public S3 Bucket to our Private S3 Bucket in this Account

Please copy the dataset into this folder on your Sagemaker Studio file system - this may take about 10-15 minutes due to the size of the data.

In [14]:
!aws s3 cp --recursive $s3_public_path_tsv/ $s3_private_path_tsv/ --exclude "*" --include "amazon_reviews_us_Digital_Software_v1_00.tsv"
!aws s3 cp --recursive $s3_public_path_tsv/ $s3_private_path_tsv/ --exclude "*" --include "amazon_reviews_us_Digital_Video_Games_v1_00.tsv"
!aws s3 cp --recursive $s3_public_path_tsv/ $s3_private_path_tsv/ --exclude "*" --include "amazon_reviews_us_Gift_Card_v1_00.tsv"

copy: s3://aai-540/amazon-reviews-pds/tsv/amazon_reviews_us_Digital_Software_v1_00.tsv to s3://sagemaker-us-east-1-944202758041/amazon-reviews-pds/tsv/amazon_reviews_us_Digital_Software_v1_00.tsv
copy: s3://aai-540/amazon-reviews-pds/tsv/amazon_reviews_us_Digital_Video_Games_v1_00.tsv to s3://sagemaker-us-east-1-944202758041/amazon-reviews-pds/tsv/amazon_reviews_us_Digital_Video_Games_v1_00.tsv
copy: s3://aai-540/amazon-reviews-pds/tsv/amazon_reviews_us_Gift_Card_v1_00.tsv to s3://sagemaker-us-east-1-944202758041/amazon-reviews-pds/tsv/amazon_reviews_us_Gift_Card_v1_00.tsv


# _Make sure ^^^^ this ^^^^ S3 COPY command above runs succesfully. We will need those datafiles for the rest of this workshop._

# List Files in our Private S3 Bucket in this Account

In [15]:
print(s3_private_path_tsv)

s3://sagemaker-us-east-1-944202758041/amazon-reviews-pds/tsv


In [16]:
!aws s3 ls $s3_private_path_tsv/

2026-09-14 00:09:47   53855391 amazon_reviews_us_Digital_Software_v1_00.tsv
2026-09-14 00:09:49   73154460 amazon_reviews_us_Digital_Video_Games_v1_00.tsv
2026-09-14 00:09:50   39977611 amazon_reviews_us_Gift_Card_v1_00.tsv


In [17]:
from IPython.core.display import display, HTML

display(
    HTML(
        '<b>Review <a target="blank" href="https://s3.console.aws.amazon.com/s3/buckets/sagemaker-{}-{}/amazon-reviews-pds/?region={}&tab=overview">S3 Bucket</a></b>'.format(
            region, account_id, region
        )
    )
)

/tmp/ipykernel_2028/4286372116.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


# Set up S3 Bucket for Homework

In [17]:
from io import BytesIO
from pathlib import Path

import boto3
import pandas as pd


bucket = "sagemaker-us-east-1-944202758041"
region = "us-east-1"

# Path the homework-2-1 dataset.csv
local_file = Path("/home/sagemaker-user/aai-540-homework/homework-2-1/data/.ipynb_checkpoints/dataset-checkpoint.csv")

s3_key = "homework-dataset/dataset.csv"
s3_uri = f"s3://{bucket}/{s3_key}"


if not local_file.is_file():
    raise FileNotFoundError(
        f"File not found: {local_file.resolve()}"
    )

print("Local file:", local_file.resolve())
print("Local file size:", local_file.stat().st_size, "bytes")


session = boto3.session.Session(region_name=region)
s3 = session.client("s3")


# Upload dataset.csv to S3 Bucket
s3.upload_file(
    Filename=str(local_file),
    Bucket=bucket,
    Key=s3_key,
)

print("Uploaded to:")
print(s3_uri)


response = s3.head_object(
    Bucket=bucket,
    Key=s3_key,
)

print("S3 object verified")
print("S3 object size:", response["ContentLength"], "bytes")


object_response = s3.get_object(
    Bucket=bucket,
    Key=s3_key,
)

df = pd.read_csv(
    BytesIO(object_response["Body"].read())
)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

display(df.head())

Local file: /home/sagemaker-user/aai-540-homework/homework-2-1/data/.ipynb_checkpoints/dataset-checkpoint.csv
Local file size: 20118244 bytes
Uploaded to:
s3://sagemaker-us-east-1-944202758041/homework-dataset/dataset.csv
S3 object verified
S3 object size: 20118244 bytes
Dataset loaded successfully
Dataset shape: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [17]:
from io import BytesIO
from pathlib import Path

import boto3
import pandas as pd


bucket = "sagemaker-us-east-1-944202758041"
region = "us-east-1"

# Path the homework-2-1 dataset.csv
local_file = Path("/home/sagemaker-user/aai-540-homework/homework-2-1/data/.ipynb_checkpoints/dataset-checkpoint.csv")

s3_key = "homework-dataset/dataset.csv"
s3_uri = f"s3://{bucket}/{s3_key}"


if not local_file.is_file():
    raise FileNotFoundError(
        f"File not found: {local_file.resolve()}"
    )

print("Local file:", local_file.resolve())
print("Local file size:", local_file.stat().st_size, "bytes")


session = boto3.session.Session(region_name=region)
s3 = session.client("s3")


# Upload dataset.csv to S3 Bucket
s3.upload_file(
    Filename=str(local_file),
    Bucket=bucket,
    Key=s3_key,
)

print("Uploaded to:")
print(s3_uri)


response = s3.head_object(
    Bucket=bucket,
    Key=s3_key,
)

print("S3 object verified")
print("S3 object size:", response["ContentLength"], "bytes")


object_response = s3.get_object(
    Bucket=bucket,
    Key=s3_key,
)

df = pd.read_csv(
    BytesIO(object_response["Body"].read())
)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

display(df.head())

Local file: /home/sagemaker-user/aai-540-homework/homework-2-1/data/.ipynb_checkpoints/dataset-checkpoint.csv
Local file size: 20118244 bytes
Uploaded to:
s3://sagemaker-us-east-1-944202758041/homework-dataset/dataset.csv
S3 object verified
S3 object size: 20118244 bytes
Dataset loaded successfully
Dataset shape: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [19]:
s3_private_path_csv = (
    "s3://sagemaker-us-east-1-944202758041/"
    "homework-dataset/dataset.csv"
)

%store s3_private_path_csv

print(s3_private_path_csv)

Stored 's3_private_path_csv' (str)
s3://sagemaker-us-east-1-944202758041/homework-dataset/dataset.csv


In [20]:
%store

Stored variables and their in-db values:
ingest_create_athena_db_passed                        -> True
ingest_create_athena_table_parquet_passed             -> True
ingest_create_athena_table_tsv_passed                 -> True
s3_private_path_csv                                   -> 's3://sagemaker-us-east-1-944202758041/homework-da
s3_private_path_tsv                                   -> 's3://sagemaker-us-east-1-944202758041/amazon-revi
s3_public_path_tsv                                    -> 's3://aai-540/amazon-reviews-pds/tsv'
setup_dependencies_passed                             -> True
setup_s3_bucket_passed                                -> True


# Release Resources

In [19]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [ ]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}